# 🧭 Atlas — Agente de IA para o benchmark GAIA

**Atlas** é um agente autônomo que responde perguntas do mundo real combinando raciocínio de um LLM com ferramentas: busca na web, leitura de páginas, Wikipedia, execução de Python, transcrição de áudio e análise de imagens.

Ele foi construído como projeto final do [Hugging Face AI Agents Course](https://huggingface.co/learn/agents-course) e avaliado em um subconjunto de **20 perguntas de nível 1 do [GAIA](https://huggingface.co/papers/2311.12983)**, um benchmark criado para medir o quanto assistentes de IA conseguem resolver tarefas que são simples para humanos (~92% de acerto), mas difíceis para modelos sozinhos (GPT-4 com plugins: ~15%).

| | |
|---|---|
| **Framework** | [smolagents](https://github.com/huggingface/smolagents) (`CodeAgent`) |
| **Modelo** | OpenAI `gpt-4o` (cérebro) + `whisper-1` (áudio) |
| **Padrão** | ReAct — ciclo *Thought → Action → Observation* |
| **Avaliação** | Exact match contra o gabarito do GAIA |
| **Meta** | ≥ 30% de acerto para o certificado do curso |

### O que este notebook mostra
1. Como montar um agente que **escreve e executa código Python** como forma de agir (CodeAgent).
2. Como criar **ferramentas próprias** (áudio, imagem, navegação web) com o decorador `@tool`.
3. Como lidar com **anexos** (planilhas, áudios, imagens, código) baixados pela API.
4. Como fazer **prompt engineering para exact match** — a resposta precisa sair exatamente no formato esperado.
5. Como rodar a avaliação completa e enviar as respostas para o leaderboard.

## 🏗️ Arquitetura

```
              ┌──────────────────────────────────────────┐
 Pergunta ──▶ │  Atlas (smolagents CodeAgent)            │
 + anexo      │                                          │
              │   Thought  → o LLM raciocina             │
              │   Action   → o LLM escreve Python        │
              │   Observation → resultado volta ao LLM   │
              │        (repete até final_answer)         │
              └──────┬───────────────────────────┬───────┘
                     │                           │
             ┌───────▼───────┐          ┌────────▼─────────┐
             │  LLM: gpt-4o  │          │   Ferramentas     │
             └───────────────┘          │ • web_search      │
                                        │ • visit_webpage   │
                                        │ • wikipedia_search│
                                        │ • transcribe_audio│
                                        │ • analyze_image   │
                                        │ • Python sandbox  │
                                        └──────────────────┘
```

**Por que um `CodeAgent`?** Em vez de chamar ferramentas com blobs JSON (uma ação por passo), o agente escreve código. Assim ele combina várias ferramentas, faz contas e manipula dados (ex.: `pandas` numa planilha) em um único passo — ideal para as perguntas do GAIA.

## 1. Instalação

In [ ]:
%pip install -q "smolagents>=1.20" openai ddgs wikipedia-api markdownify pandas openpyxl huggingface_hub requests

## 2. Configuração das chaves

O notebook precisa de:
- **`OPENAI_API_KEY`** — para o `gpt-4o` e o `whisper-1` (API paga; uma rodada completa custa menos de US$ 1).
- **`HF_TOKEN`** *(opcional)* — para baixar anexos direto do dataset [`gaia-benchmark/GAIA`](https://huggingface.co/datasets/gaia-benchmark/GAIA) quando a API do curso falha. É preciso aceitar os termos do dataset antes.

> As chaves são lidas de variáveis de ambiente ou digitadas na hora — **nunca** ficam salvas no notebook.

In [ ]:
import os
from getpass import getpass

for key in ["OPENAI_API_KEY", "HF_TOKEN"]:
    if not os.getenv(key):
        value = getpass(f"{key} (Enter para pular): ")
        if value:
            os.environ[key] = value

print("OPENAI_API_KEY:", "ok" if os.getenv("OPENAI_API_KEY") else "FALTANDO")
print("HF_TOKEN:", "ok" if os.getenv("HF_TOKEN") else "não definido (opcional)")

## 3. Imports e constantes

In [ ]:
import os
import re
import base64
import tempfile

import requests
import pandas as pd
from markdownify import markdownify
from openai import OpenAI
from huggingface_hub import hf_hub_download
from smolagents import (
    CodeAgent,
    OpenAIServerModel,
    DuckDuckGoSearchTool,
    WikipediaSearchTool,
    tool,
)

API_URL = "https://agents-course-unit4-scoring.hf.space"  # API de avaliação do curso
MODEL_ID = "gpt-4o"
MAX_STEPS = 12

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 4. Ferramentas

No smolagents, uma ferramenta é só uma função Python com:
- o decorador **`@tool`**;
- **type hints** nos argumentos e no retorno;
- uma **docstring com `Args:`** — é ela que o LLM lê para decidir *quando* e *como* usar a ferramenta (ele nunca vê o código).

Além das ferramentas prontas (`DuckDuckGoSearchTool`, `WikipediaSearchTool`), o Atlas tem três ferramentas próprias:

| Ferramenta | Por que existe |
|---|---|
| `visit_webpage` | A versão padrão era bloqueada (HTTP 403) pela Wikipedia por não enviar `User-Agent`. Esta envia um cabeçalho identificado e converte o HTML em markdown. |
| `transcribe_audio` | Perguntas com anexos `.mp3` (receitas, gravações de aula). Usa o Whisper. |
| `analyze_image` | Perguntas com imagens (ex.: posição de xadrez). Usa a visão multimodal do `gpt-4o`. |

In [ ]:
@tool
def visit_webpage(url: str) -> str:
    """Opens a webpage and returns its content as markdown text.

    Args:
        url: The full URL of the page to open.
    """
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Atlas-GAIA-agent/1.0)"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return markdownify(r.text)[:40000]


@tool
def transcribe_audio(file_path: str) -> str:
    """Transcribes an audio file (mp3, wav, m4a) to text.

    Args:
        file_path: Local path to the audio file.
    """
    with open(file_path, "rb") as f:
        result = client.audio.transcriptions.create(model="whisper-1", file=f)
    return result.text


@tool
def analyze_image(file_path: str, question: str) -> str:
    """Looks at an image file and answers a question about it.

    Args:
        file_path: Local path to the image (png, jpg).
        question: What you want to know about the image.
    """
    ext = file_path.split(".")[-1].lower()
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(file_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": question},
                {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
            ],
        }],
    )
    return resp.choices[0].message.content

## 5. Prompt: instruções para *exact match*

A correção do GAIA é por **exact match**: `"3"` está certo, `"3 albums"` ou `"The answer is 3."` estão errados. Por isso as instruções focam no **formato da resposta**, baseadas no prompt oficial do GAIA (sem o prefixo `FINAL ANSWER`, como pede o curso).

Note que **não é preciso explicar o ciclo Thought → Action → Observation**: o `CodeAgent` já tem um *system prompt* interno com essas regras e exemplos *few-shot*. Estas instruções entram como a **tarefa** e só tratam do que é específico do GAIA.

In [ ]:
INSTRUCTIONS = """You are a general AI assistant solving a benchmark question.
Think step by step and use your tools (web search, visit webpages, Wikipedia, Python,
audio transcription, image analysis) to find the correct answer. Verify facts.

When you call final_answer, give ONLY the answer, nothing else:
- A number: digits only, no commas as thousands separators, no units ($, %) unless asked.
- A string: as few words as possible, no articles, no abbreviations.
- A list: a single comma-separated string, following the rules above for each element.
- Follow any format the question asks for exactly (e.g. alphabetical order, plural form).
- Never write 'FINAL ANSWER' or explanations.
- Never invent or simulate data, files or transcriptions. If a file path is given, use exactly that path.
- If a webpage is blocked, try another source instead of guessing.
- In each Thought, check whether the last Observation actually answers the question before calling final_answer."""

## 6. O agente Atlas

A classe `AtlasAgent` encapsula três responsabilidades:

1. **Montar o `CodeAgent`** com o modelo, as ferramentas e os imports liberados no sandbox (`pandas`, `numpy`, ...). O sandbox bloqueia qualquer import fora dessa lista.
2. **Baixar anexos**: primeiro pela API do curso; se falhar (a rota às vezes devolve 404), pelo dataset oficial do GAIA no Hugging Face.
3. **Normalizar a resposta**: se o agente devolver uma lista Python (`['a', 'b']`), ela vira `a, b`; prefixos como `FINAL ANSWER:` são removidos.

`code_block_tags="markdown"` faz o smolagents aceitar código em blocos markdown — o `gpt-4o` às vezes não usa as tags `<code>` esperadas, o que desperdiçava passos com erros de *parsing*.

In [ ]:
class AtlasAgent:
    def __init__(self, model_id: str = MODEL_ID, max_steps: int = MAX_STEPS):
        model = OpenAIServerModel(
            model_id=model_id,
            api_key=os.getenv("OPENAI_API_KEY"),
            client_kwargs={"timeout": 60, "max_retries": 2},
        )
        agent_kwargs = dict(
            model=model,
            tools=[
                DuckDuckGoSearchTool(),
                visit_webpage,
                WikipediaSearchTool(user_agent="Atlas-GAIA-agent (portfolio project)"),
                transcribe_audio,
                analyze_image,
            ],
            additional_authorized_imports=[
                "pandas", "numpy", "openpyxl", "json", "re", "math",
                "datetime", "collections", "itertools", "statistics",
            ],
            max_steps=max_steps,
        )
        try:
            self.agent = CodeAgent(**agent_kwargs, code_block_tags="markdown")
        except TypeError:  # versões antigas do smolagents não têm esse parâmetro
            self.agent = CodeAgent(**agent_kwargs)

    # ---------- anexos ----------
    def download_file(self, task_id: str, file_name: str) -> str | None:
        try:
            r = requests.get(f"{API_URL}/files/{task_id}", timeout=30)
            r.raise_for_status()
            path = os.path.join(tempfile.gettempdir(), file_name)
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        except Exception as e:
            print(f"  API download failed ({e}); trying GAIA dataset...")
        try:
            return hf_hub_download(
                repo_id="gaia-benchmark/GAIA",
                repo_type="dataset",
                filename=f"2023/validation/{file_name}",
                token=os.getenv("HF_TOKEN"),
            )
        except Exception as e:
            print(f"  GAIA dataset download failed: {e}")
            return None

    # ---------- resposta ----------
    @staticmethod
    def normalize(result) -> str:
        if isinstance(result, (list, tuple)):
            answer = ", ".join(str(x) for x in result)
        else:
            answer = str(result).strip()
        return re.sub(r"(?i)^final answer:\s*", "", answer).strip()

    def __call__(self, question: str, task_id: str | None = None, file_name: str | None = None) -> str:
        prompt = INSTRUCTIONS + "\n\nQuestion: " + question
        if file_name and task_id:
            path = self.download_file(task_id, file_name)
            if path:
                prompt += (
                    f"\n\nThe question has an attached file saved at: {path}\n"
                    "Use Python to read spreadsheets/code/text files, "
                    "transcribe_audio for audio and analyze_image for images."
                )
            else:
                prompt += "\n\nThe attached file could not be downloaded. Do not invent its content."
        try:
            return self.normalize(self.agent.run(prompt))
        except Exception as e:
            return f"AGENT ERROR: {e}"


atlas = AtlasAgent()
print("Atlas pronto ✅")

### Espiando o *system prompt* interno

Para ver como o smolagents ensina o ciclo ReAct ao modelo (as regras de *Thought*, *Code* e *Observation*, a lista de ferramentas e os exemplos):

In [ ]:
print(atlas.agent.prompt_templates["system_prompt"][:3000])

## 7. Teste rápido com uma pergunta aleatória

Antes de rodar as 20 perguntas, vale testar uma. Os logs do smolagents mostram cada passo: o raciocínio, o código gerado e a observação devolvida.

In [ ]:
item = requests.get(f"{API_URL}/random-question", timeout=30).json()
print("Pergunta:", item["question"])
print("Anexo:", item.get("file_name") or "-")

answer = atlas(item["question"], item["task_id"], item.get("file_name"))
print("\nResposta do Atlas:", answer)

## 8. Avaliação completa

Roda o agente nas 20 perguntas. Leva de 10 a 30 minutos, pois cada pergunta pode ter até 12 passos.

In [ ]:
questions = requests.get(f"{API_URL}/questions", timeout=30).json()
print(f"{len(questions)} perguntas")

results = []
for i, item in enumerate(questions, 1):
    print(f"\n[{i}/{len(questions)}] {item['question'][:100]}...")
    answer = atlas(item["question"], item["task_id"], item.get("file_name"))
    print("  ->", answer)
    results.append({
        "task_id": item["task_id"],
        "question": item["question"],
        "file_name": item.get("file_name") or "",
        "submitted_answer": answer,
    })

df = pd.DataFrame(results)
df.to_csv("atlas_answers.csv", index=False)
df[["question", "submitted_answer"]]

## 9. Envio para o leaderboard *(opcional)*

A API pede:
- **`username`** — seu usuário do Hugging Face;
- **`agent_code`** — link público para o código do agente (ex.: o Space ou este repositório no GitHub);
- **`answers`** — lista de `{"task_id", "submitted_answer"}`.

Deixe `SUBMIT = False` se quiser só olhar as respostas.

In [ ]:
SUBMIT = False
HF_USERNAME = "seu-usuario-hf"
AGENT_CODE_URL = "https://huggingface.co/spaces/seu-usuario-hf/seu-space/tree/main"

if SUBMIT:
    payload = {
        "username": HF_USERNAME,
        "agent_code": AGENT_CODE_URL,
        "answers": [{"task_id": r["task_id"], "submitted_answer": r["submitted_answer"]} for r in results],
    }
    resp = requests.post(f"{API_URL}/submit", json=payload, timeout=120)
    resp.raise_for_status()
    d = resp.json()
    print(f"Score: {d.get('score')}% ({d.get('correct_count')}/{d.get('total_attempted')} corretas)")
    print(d.get("message", ""))
else:
    print("Envio desativado (SUBMIT = False).")

## 10. Aprendizados

Problemas reais encontrados durante o desenvolvimento e como foram resolvidos:

| Problema observado nos logs | Causa | Solução no Atlas |
|---|---|---|
| `403 Forbidden` ao abrir páginas da Wikipedia | Requisição sem `User-Agent` | Ferramenta `visit_webpage` própria com cabeçalho identificado |
| `404` ao baixar anexos pela API do curso | Rota `/files/{task_id}` instável | Plano B: `hf_hub_download` do dataset oficial do GAIA |
| Agente **inventava** transcrições e números quando o arquivo faltava | LLM tentando "ajudar" | Regra explícita no prompt: nunca simular dados; aviso quando o download falha |
| Resposta `['broccoli', 'celery']` | `final_answer` recebeu uma lista Python | Normalização para `broccoli, celery` |
| `Error in code parsing` em muitos passos | Modelo não usava as tags `<code>` | `code_block_tags="markdown"` |
| Requisição travada por minutos | Sem timeout no cliente da OpenAI | `client_kwargs={"timeout": 60, "max_retries": 2}` |

### Conceitos aplicados
- **Agente = LLM (cérebro) + ferramentas (corpo)**: o LLM não sabe o que está na web hoje nem ouve um `.mp3`; as ferramentas resolvem isso.
- **ReAct**: o modelo intercala raciocínio e ação e se adapta a cada observação (ex.: uma busca vazia leva a uma nova estratégia).
- **Ações em código vs. JSON**: código permite loops, variáveis e várias ferramentas num passo.
- **Multimodalidade**: texto, imagem (base64 para o `gpt-4o`) e áudio (Whisper).
- **Prompt engineering para avaliação automática**: formato rígido de saída para *exact match*.

## 11. Próximos passos
- Ferramenta para **vídeos do YouTube** (transcrição/legendas), hoje o principal ponto cego.
- **Sistema multi-agente**: um gerente delegando para um agente de pesquisa e outro de arquivos.
- Motor de xadrez (ex.: `python-chess` + Stockfish) para perguntas de tabuleiro.
- Busca com uma API mais estável (Tavily/Serper) no lugar do DuckDuckGo, que sofre *rate limit*.